# EDA für das Regressionsmodell `vehicle_claim`

Dieses Notebook verändert oder löscht keine Daten — es zeigt nur Statistiken
und Diagramme an, damit Sie vor einer Entscheidung über Ausreißer eine
fundierte Grundlage haben.

Im Projekt-Root ausführen (dort, wo `src.ml_config` und
`data/raw/dataset.csv` verfügbar sind).

In [ ]:
import sys
import os
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
os.chdir(project_root)

print("Project root:", project_root)
print("CWD now:", os.getcwd())

Project root: d:\visual-code-projects\InterGeeks-Agiles-Programmierprojekt
CWD теперь: d:\visual-code-projects\InterGeeks-Agiles-Programmierprojekt


In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.ml_config import (
    INSURANCE_DATASET_PATH,
    TARGET_FIELDS,
    PREDICTION_FIELD,
    FIELDS_TO_DELETE,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

OUT_DIR = Path("data/output/eda")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [20]:
import os
print("CWD:", os.getcwd())
print(os.listdir())
INSURANCE_DATASET_PATH = Path("./data/raw/dataset.csv")
if not INSURANCE_DATASET_PATH.exists():
    raise FileNotFoundError(f"Datei nicht gefunden: {INSURANCE_DATASET_PATH}")

df = pd.read_csv(INSURANCE_DATASET_PATH)
print(f"Zeilen: {len(df)}, Spalten: {df.shape[1]}")
df.head()

CWD: d:\visual-code-projects\InterGeeks-Agiles-Programmierprojekt
['.env', '.git', '.gitignore', '.pre-commit-config.yaml', '.streamlit', '.venv', 'app', 'checkpoints', 'data', 'docs', 'main.py', 'network_config.json', 'README.md', 'requirements.txt', 'sql', 'src', '__pycache__']
Zeilen: 1000, Spalten: 40


,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,insured_sex,insured_education_level,insured_occupation,insured_hobbies,insured_relationship,capital-gains,capital-loss,incident_date,incident_type,collision_type,incident_severity,authorities_contacted,incident_state,incident_city,incident_location,incident_hour_of_the_day,number_of_vehicles_involved,property_damage,bodily_injuries,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,_c39
0,328,48,521585,2014-10-17,OH,250/500,1000,1406.91,0,466132,MALE,MD,craft-repair,sleeping,husband,53300,0,2015-01-25,Single Vehicle Collision,Side Collision,Major Damage,Police,SC,Columbus,9935 4th Drive,5,1,YES,1,2,YES,71610,6510,13020,52080,Saab,92x,2004,Y,NaN
1,228,42,342868,2006-06-27,IN,250/500,2000,1197.22,5000000,468176,MALE,MD,machine-op-inspct,reading,other-relative,0,0,2015-01-21,Vehicle Theft,?,Minor Damage,Police,VA,Riverwood,6608 MLK Hwy,8,1,?,0,0,?,5070,780,780,3510,Mercedes,E400,2007,Y,NaN
2,134,29,687698,2000-09-06,OH,100/300,2000,1413.14,5000000,430632,FEMALE,PhD,sales,board-games,own-child,35100,0,2015-02-22,Multi-vehicle Collision,Rear Collision,Minor Damage,Police,NY,Columbus,7121 Francis Lane,7,3,NO,2,3,NO,34650,7700,3850,23100,Dodge,RAM,2007,N,NaN
3,256,41,227811,1990-05-25,IL,250/500,2000,1415.74,6000000,608117,FEMALE,PhD,armed-forces,board-games,unmarried,48900,-62400,2015-01-10,Single Vehicle Collision,Front Collision,Major Damage,Police,OH,Arlington,6956 Maple Drive,5,1,?,1,2,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y,NaN
4,228,44,367455,2014-06-06,IL,500/1000,1000,1583.91,6000000,610706,MALE,Associate,sales,board-games,unmarried,66000,-46000,2015-02-17,Vehicle Theft,?,Minor Damage,NaN,NY,Arlington,3041 3rd Ave,20,1,NO,0,1,NO,6500,1300,650,4550,Accura,RSX,2009,N,NaN


## 1. Allgemeine Informationen zum Datensatz

In [ ]:
df.dtypes

## 2. Zielvariable: `vehicle_claim` — Basisstatistik

In [ ]:
if PREDICTION_FIELD not in df.columns:
    raise KeyError(f"Spalte {PREDICTION_FIELD} ist im Datensatz nicht vorhanden")

y = df[PREDICTION_FIELD]
y.describe()

In [ ]:
print(f"Fehlende Werte in {PREDICTION_FIELD}: {y.isna().sum()}")
print(f"Nullwerte: {(y == 0).sum()} ({(y == 0).mean():.1%})")
print(f"Negative Werte: {(y < 0).sum()}")

In [ ]:
# IQR-Methode als Orientierung (nicht zum automatischen Entfernen!)
q1, q3 = y.quantile(0.25), y.quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
n_outliers = ((y < low) | (y > high)).sum()

print(f"IQR-Grenzen: [{low:,.0f}, {high:,.0f}]")
print(f"Punkte außerhalb von IQR*1.5: {n_outliers} ({n_outliers/len(y):.1%})")

In [ ]:
# Perzentile, um die Verteilungsenden (Tails) zu sehen
for p in [0.01, 0.05, 0.5, 0.95, 0.99]:
    print(f"  Perzentil {p:>4.0%}: {y.quantile(p):,.0f}")

### Visuelle Prüfung: Verteilung und Boxplot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y, bins=50)
axes[0].set_title("Verteilung von vehicle_claim")
axes[0].set_xlabel("vehicle_claim")

axes[1].boxplot(y.dropna(), vert=False)
axes[1].set_title("Boxplot vehicle_claim")

fig.tight_layout()
fig.savefig(OUT_DIR / "vehicle_claim_distribution.png", dpi=120)
plt.show()

## 3. `vehicle_claim` nach `incident_severity`

Wenn bei `Trivial Damage` (oder einer ähnlichen Kategorie) die meisten Werte
`0` sind, handelt es sich nicht um Ausreißer, sondern um einen eigenen
Modus: Das Modell sollte dies als legitimen "Nullfall" erkennen und nicht
als zu entfernendes Rauschen.

In [ ]:
if "incident_severity" in df.columns:
    display(
        df.groupby("incident_severity")[PREDICTION_FIELD]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .sort_values("mean")
    )

In [ ]:
if "incident_severity" in df.columns:
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    df.boxplot(column=PREDICTION_FIELD, by="incident_severity", ax=ax2, rot=30)
    ax2.set_title("vehicle_claim nach incident_severity")
    plt.suptitle("")
    fig2.tight_layout()
    fig2.savefig(OUT_DIR / "vehicle_claim_by_severity.png", dpi=120)
    plt.show()

## 4. `vehicle_claim` nach `collision_type`

In [ ]:
if "collision_type" in df.columns:
    display(
        df.groupby("collision_type", dropna=False)[PREDICTION_FIELD]
        .agg(["count", "mean", "median", "std"])
        .sort_values("mean")
    )

## 5. `vehicle_claim` nach `incident_type`

In [ ]:
if "incident_type" in df.columns:
    display(
        df.groupby("incident_type", dropna=False)[PREDICTION_FIELD]
        .agg(["count", "mean", "median", "std"])
        .sort_values("mean")
    )

## 6. Fehlende Werte in `TARGET_FIELDS` + `PREDICTION_FIELD`

In [ ]:
cols = TARGET_FIELDS + [PREDICTION_FIELD]
cols = [c for c in cols if c in df.columns]
miss = df[cols].isna().sum()
miss = miss[miss > 0]

if len(miss):
    print(miss)
else:
    print("Keine fehlenden Werte (prüfen Sie aber auch Platzhalter wie '?' / 'NA' /")
    print("'UNKNOWN' — in dieser Art von Datensatz (insurance fraud) sind fehlende")
    print("Werte häufig als '?' kodiert.")

### 6b. Prüfung auf Platzhalter für fehlende Werte (`?`)

In [ ]:
for c in cols:
    if df[c].dtype == object:
        n_q = (df[c] == "?").sum()
        if n_q:
            print(f"  {c}: {n_q} Werte mit '?'")

## 7. Korrelation der numerischen Merkmale mit der Zielvariable

Wenn alle Korrelationen nahe 0 liegen, bestätigt das: Das aktuelle
Merkmalsset erklärt `vehicle_claim` nur schwach, und das niedrige R² ist
eher eine Folge der Merkmalsauswahl als nur von Ausreißern.

In [ ]:
numeric_candidates = (
    [c for c in TARGET_FIELDS if pd.api.types.is_numeric_dtype(df[c])]
    if all(c in df.columns for c in TARGET_FIELDS)
    else []
)
numeric_candidates = [c for c in numeric_candidates if c in df.columns]

if numeric_candidates:
    corr = df[numeric_candidates + [PREDICTION_FIELD]].corr(numeric_only=True)[PREDICTION_FIELD]
    print(corr.sort_values(ascending=False))

## 8. Entfernte Felder (`FIELDS_TO_DELETE`) — mögliches Leakage vs. nützliches Signal

`total_claim_amount` korreliert vermutlich stark mit `vehicle_claim`, da
häufig `total = injury + property + vehicle` gilt. Das ist direktes
Leakage — es ist richtig, dass Sie es aus den Merkmalen entfernt haben.

`injury_claim` und `property_claim` sind separate Komponenten ohne klaren
arithmetischen Bezug zu `vehicle_claim`. Sie könnten daher als Merkmale
**beibehalten** werden (kein Leakage), sofern sie zum Vorhersagezeitpunkt
bereits bekannt sind.

In [ ]:
leak_candidates = ["injury_claim", "total_claim_amount", "property_claim"]
present = [c for c in leak_candidates if c in df.columns]

if present:
    corr2 = df[present + [PREDICTION_FIELD]].corr(numeric_only=True)[PREDICTION_FIELD]
    print(corr2)

## Fazit

Schauen Sie sich die Diagramme unter `data/output/eda/` sowie die
Abschnitte 2, 3 und 7 oben an, bevor Sie über das Entfernen von Ausreißern
entscheiden.